In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
CV_FOLDS     = 5

# ── Load preprocessed data ────────────────────────────────────────────────────
X_train       = joblib.load('../outputs/X_train.pkl')
X_test        = joblib.load('../outputs/X_test.pkl')
y_train       = joblib.load('../outputs/y_train.pkl')
y_test        = joblib.load('../outputs/y_test.pkl')
feature_names = joblib.load('../outputs/feature_names.pkl')

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {len(feature_names)}')
print('Data loaded ✓')

In [ ]:
# ── Helper: evaluate any model cleanly ───────────────────────────────────────
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name, cv=CV_FOLDS):
    """Train, cross-validate, and return a metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    cv_scores = cross_val_score(
        model, X_tr, y_tr,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1'
    )

    metrics = {
        'model'    : name,
        'accuracy' : accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall'   : recall_score(y_te, y_pred),
        'f1'       : f1_score(y_te, y_pred),
        'roc_auc'  : roc_auc_score(y_te, y_proba),
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std' : cv_scores.std(),
    }

    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'  Accuracy  : {metrics["accuracy"]:.4f}')
    print(f'  Precision : {metrics["precision"]:.4f}')
    print(f'  Recall    : {metrics["recall"]:.4f}')
    print(f'  F1 Score  : {metrics["f1"]:.4f}')
    print(f'  ROC-AUC   : {metrics["roc_auc"]:.4f}')
    print(f'  CV F1     : {metrics["cv_f1_mean"]:.4f} ± {metrics["cv_f1_std"]:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_te, y_pred, target_names=['Retained', 'Churned']))

    return model, metrics, y_pred, y_proba

print('Helper defined ✓')

In [ ]:
# ── Model 1: Logistic Regression (baseline) ───────────────────────────────────
# max_iter=1000 ensures convergence on this dataset
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight='balanced'   # handles the 73/27 class imbalance
)

lr_model, lr_metrics, lr_pred, lr_proba = evaluate_model(
    lr_model, X_train, y_train, X_test, y_test,
    name='Logistic Regression'
)

In [ ]:
# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1   # use all CPU cores
)

rf_model, rf_metrics, rf_pred, rf_proba = evaluate_model(
    rf_model, X_train, y_train, X_test, y_test,
    name='Random Forest'
)

In [ ]:
# ── Model 3: Random Forest with GridSearchCV (tuned) ─────────────────────────
print('Running GridSearchCV — this takes ~2-3 minutes...')

param_grid = {
    'n_estimators' : [100, 200],
    'max_depth'    : [6, 10, None],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV F1 : {grid_search.best_score_:.4f}')

rf_tuned = grid_search.best_estimator_
rf_tuned, rf_tuned_metrics, rf_tuned_pred, rf_tuned_proba = evaluate_model(
    rf_tuned, X_train, y_train, X_test, y_test,
    name='Random Forest (tuned)'
)

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
results_df = pd.DataFrame([lr_metrics, rf_metrics, rf_tuned_metrics])
results_df = results_df.set_index('model')
results_df = results_df[['accuracy','precision','recall','f1','roc_auc','cv_f1_mean']]
results_df.columns = ['Accuracy','Precision','Recall','F1','ROC-AUC','CV F1']

print('\n=== Model Comparison ===')
print(results_df.round(4).to_string())

best_model_name = results_df['F1'].idxmax()
print(f'\nBest model by F1: {best_model_name}')

In [ ]:
# ── Save all models ───────────────────────────────────────────────────────────
joblib.dump(lr_model,   '../models/logistic_regression.pkl')
joblib.dump(rf_model,   '../models/random_forest.pkl')
joblib.dump(rf_tuned,   '../models/random_forest_tuned.pkl')
joblib.dump(results_df, '../outputs/model_comparison.pkl')

# Save predictions for evaluation notebook
preds = {
    'lr_pred'         : lr_pred,         'lr_proba'         : lr_proba,
    'rf_pred'         : rf_pred,         'rf_proba'         : rf_proba,
    'rf_tuned_pred'   : rf_tuned_pred,   'rf_tuned_proba'   : rf_tuned_proba,
}
joblib.dump(preds, '../outputs/predictions.pkl')

print('Models saved to predictive-modeling/models/ :')
print('  logistic_regression.pkl')
print('  random_forest.pkl')
print('  random_forest_tuned.pkl')
print('  scaler.pkl  (from notebook 01)')
print('\nNext → open 03_evaluation.ipynb')